In [ ]:
# ==============================================================================
# File: 03_Evaluate_and_Plot.ipynb (Cell 1 of 4)
# What this notebook does: Mounts Google Drive, clones/syncs https://github.com/Alekhagrawal/ZeroFake.git,
#   unzips cropped_faces.zip onto Colab's fast local SSD (/content/data), runs evaluate.py across
#   all trained checkpoints to generate ablation_summary.csv, robustness_summary.csv, and all 4
#   300-DPI PNG charts, and backs up /content/results/ permanently into Google Drive.
# Inputs: /content/drive/MyDrive/ZeroFake_Storage/{cropped_faces.zip, checkpoints/}.
# Outputs: /content/drive/MyDrive/ZeroFake_Storage/results/ (CSVs + 300-DPI PNG plots).
# ==============================================================================

# --- CELL 1: Mount Google Drive, Pull Latest GitHub Repo & Install Dependencies ---
import os
import shutil
import sys
from google.colab import drive

# 1. Safely clear any unmounted placeholder /content/drive directory before mounting
if not os.path.ismount('/content/drive'):
    shutil.rmtree('/content/drive', ignore_errors=True)
drive.mount('/content/drive')

# 2. Clone or force-sync https://github.com/Alekhagrawal/ZeroFake.git into /content/ZeroFake
GITHUB_REPO_URL = 'https://github.com/Alekhagrawal/ZeroFake.git'
REPO_DIR = '/content/ZeroFake'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print(f'[INFO] Fetching and syncing latest commits from {GITHUB_REPO_URL}...')
    !git -C /content/ZeroFake fetch origin
    !git -C /content/ZeroFake reset --hard origin/main
else:
    print(f'[INFO] Cloning {GITHUB_REPO_URL} into {REPO_DIR}...')
    !rm -rf /content/ZeroFake
    !git clone https://github.com/Alekhagrawal/ZeroFake.git /content/ZeroFake

%cd /content/ZeroFake
os.environ['PYTHONPATH'] = '/content/ZeroFake'
if '/content/ZeroFake' not in sys.path:
    sys.path.insert(0, '/content/ZeroFake')

# 3. Install facenet-pytorch with --no-deps first (avoiding Pillow 10.2.0 compilation on Python 3.13)
#    and install remaining packages using 'grad-cam'
!pip install -q --no-deps facenet-pytorch
!pip install -q timm opencv-python-headless scikit-learn matplotlib seaborn grad-cam gradio pyyaml tqdm

# 4. Pre-flight CPU dry-run verification
!PYTHONPATH=/content/ZeroFake python -u evaluate.py --dry-run
if _exit_code != 0:
    raise RuntimeError(f'evaluate.py --dry-run failed with exit code {_exit_code}')
print('[SUCCESS] Google Drive mounted, repo synced at /content/ZeroFake, and evaluate.py --dry-run verified!')


In [ ]:
# --- CELL 2: Unzip /content/drive/MyDrive/ZeroFake_Storage/cropped_faces.zip onto /content/data ---
# Plain-English explanation:
#   Unzips your cropped 224x224 face dataset onto Colab's local SSD (/content/data) so
#   evaluation, robustness stress-testing, t-SNE, and Grad-CAM run at full GPU speed.

import os
import shutil
import zipfile

STORAGE_DIR = '/content/drive/MyDrive/ZeroFake_Storage'
ZIP_PATH = os.path.join(STORAGE_DIR, 'cropped_faces.zip')
LOCAL_DATA_DIR = '/content/data'
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f'cropped_faces.zip not found at {ZIP_PATH}!')

crops_dir = os.path.join(LOCAL_DATA_DIR, 'crops')
if not os.path.exists(crops_dir):
    print(f'[INFO] Unzipping {ZIP_PATH} onto local SSD {LOCAL_DATA_DIR}...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(LOCAL_DATA_DIR)
    print('[SUCCESS] Extracted cropped faces to /content/data.')
else:
    print('[INFO] Cropped faces already present in /content/data/crops.')

for fname in ('splits.json', 'frames_manifest.csv', 'failed_detections.csv'):
    drive_file = os.path.join(STORAGE_DIR, fname)
    local_file = os.path.join(LOCAL_DATA_DIR, fname)
    if os.path.exists(drive_file) and not os.path.exists(local_file):
        shutil.copy2(drive_file, local_file)

# Flatten nested processed_faces/ folder if present
nested_dir = os.path.join(LOCAL_DATA_DIR, 'processed_faces')
if os.path.exists(nested_dir) and not os.path.exists(os.path.join(LOCAL_DATA_DIR, 'frames_manifest.csv')):
    for item in os.listdir(nested_dir):
        src_item = os.path.join(nested_dir, item)
        dst_item = os.path.join(LOCAL_DATA_DIR, item)
        if not os.path.exists(dst_item):
            shutil.move(src_item, dst_item)

assert os.path.exists(os.path.join(LOCAL_DATA_DIR, 'frames_manifest.csv')), 'frames_manifest.csv missing in /content/data!'
print(f'[READY] Local SSD dataset directory contents: {os.listdir(LOCAL_DATA_DIR)}')


In [ ]:
# --- CELL 3: Run evaluate.py Across All Saved Checkpoints -> /content/results/ ---
# Plain-English explanation:
#   Evaluates all saved checkpoints under /content/drive/MyDrive/ZeroFake_Storage/checkpoints/
#   on test_closed and test_unseen, runs the 13-condition in-memory robustness stress tests,
#   and generates all 4 300-DPI PNG charts (ROC comparison, Confusion Matrices, t-SNE, Grad-CAM)
#   inside /content/results/.

import os

%cd /content/ZeroFake
CHECKPOINTS_DIR = '/content/drive/MyDrive/ZeroFake_Storage/checkpoints'
LOCAL_RESULTS_DIR = '/content/results'
os.makedirs(LOCAL_RESULTS_DIR, exist_ok=True)

print(f'[EVAL] Running evaluate.py on checkpoints in {CHECKPOINTS_DIR}...')
!PYTHONPATH=/content/ZeroFake python -u evaluate.py --checkpoints_dir /content/drive/MyDrive/ZeroFake_Storage/checkpoints --data_dir /content/data --output_dir /content/results --batch_size 64
if _exit_code != 0:
    raise RuntimeError(f'evaluate.py failed with exit code {_exit_code}')
print(f'[SUCCESS] All CSV summaries and 300-DPI PNG plots saved to {LOCAL_RESULTS_DIR}!')


In [ ]:
# --- CELL 4: Copy Entire /content/results/ Folder Permanently to Google Drive ---
# Plain-English explanation:
#   Copies ablation_summary.csv, results_summary.csv, robustness_summary.csv, and all 4
#   high-resolution 300-DPI PNG charts from /content/results/ into
#   /content/drive/MyDrive/ZeroFake_Storage/results/ so they stay permanently backed up.

import os
import shutil

LOCAL_RESULTS_DIR = '/content/results'
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/ZeroFake_Storage/results'

if not os.path.exists(LOCAL_RESULTS_DIR):
    raise FileNotFoundError(f'Local results folder not found: {LOCAL_RESULTS_DIR}')

shutil.copytree(LOCAL_RESULTS_DIR, DRIVE_RESULTS_DIR, dirs_exist_ok=True)

print(f'[SUCCESS] Copied all evaluation CSVs and 300-DPI PNG plots to: {DRIVE_RESULTS_DIR}')
for root, _, files in os.walk(DRIVE_RESULTS_DIR):
    for f in sorted(files):
        full_p = os.path.join(root, f)
        rel_p = os.path.relpath(full_p, DRIVE_RESULTS_DIR)
        kb = os.path.getsize(full_p) / 1024.0
        print(f'  - {rel_p:<35} ({kb:.1f} KB)')
